# CohortLint public-data demonstration

This executable notebook downloads sample metadata from the public NCBI GEO series [GSE171524](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE171524). It aggregates cell-level rows to donors and deterministically partitions donors into two demonstration cohorts. The partition is synthetic and is not a claim about the study design; it lets the complete multi-cohort workflow be reproduced without patient data or expression matrices.

In [ ]:
%pip install -q cohortlint

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
import yaml

work = Path("demo_output")
work.mkdir(exist_ok=True)
url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE171nnn/"
    "GSE171524/suppl/GSE171524_lung_metaData.txt.gz"
)
source = work / "GSE171524_lung_metaData.txt.gz"
if not source.exists():
    urlretrieve(url, source)

cells = pd.read_csv(source, sep="\t", skiprows=[1], low_memory=False)
donors = (
    cells[["donor_id", "age", "sex", "group"]]
    .drop_duplicates(subset="donor_id")
    .rename(columns={"donor_id": "sample_id", "group": "condition"})
    .sort_values("sample_id")
    .reset_index(drop=True)
)
donors["batch"] = donors.index.map(lambda index: f"library_{index % 2 + 1}")
donors["cohort"] = donors.index.map(lambda index: f"demo_{index % 2 + 1}")
for cohort, frame in donors.groupby("cohort"):
    frame.drop(columns="cohort").to_csv(work / f"{cohort}.csv", index=False)
donors.head()

In [ ]:
config = {
    "version": 1,
    "cohorts": [
        {"name": name, "path": str((work / f"{name}.csv").resolve()), "sample_id": "sample_id"}
        for name in ("demo_1", "demo_2")
    ],
    "schema": {
        "age": {"type": "numeric", "unit": "years", "range": [0, 120], "required": True},
        "sex": {"type": "categorical", "allowed": ["male", "female"], "required": True},
        "condition": {"type": "categorical", "role": "biological", "required": True},
        "batch": {"type": "categorical", "role": "technical", "required": True},
    },
    "privacy": {"k_anonymity_threshold": 5},
    "output": {"lang": "en"},
}
config_path = work / "cohortlint.yaml"
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-m", "cohortlint", "check",
        "--config", str(config_path), "--format", "html",
        "--output", str(work / "report.html"), "--fail-on", "never",
    ],
    check=True,
    text=True,
    capture_output=True,
)
print(result.stdout or f"Wrote {work / 'report.html'}")